In [1]:
import numpy as np
import torch
from huggingface_hub import hf_hub_download
from src.models.lrm_mesh import InstantMesh
from src.utils.camera_util import get_zero123plus_input_cameras
from src.utils.mesh_util import save_obj_with_mtl
from PIL import Image
from torchvision.transforms import v2
import trimesh
import io

device = torch.device("cuda")

In [12]:
model_ckpt_path = hf_hub_download(
    repo_id="TencentARC/InstantMesh",
    filename="instant_mesh_large.ckpt",
    repo_type="model"
)
model = InstantMesh(
    encoder_feat_dim=768,
    encoder_freeze=False,
    encoder_model_name="facebook/dino-vitb16",
    transformer_dim=1024,
    transformer_layers=16,
    transformer_heads=16,
    triplane_low_res=32,
    triplane_high_res=64,
    triplane_dim=80,
    rendering_samples_per_ray=128,
    grid_res=128,
    grid_scale=2.1
)
state_dict = torch.load(model_ckpt_path, map_location="cpu")["state_dict"]
state_dict = {
    k[14:]: v
    for k, v in state_dict.items()
    if k.startswith("lrm_generator.") and "source_camera" not in k
}
model.load_state_dict(state_dict, strict=True)
model = model.to(device)
model.init_flexicubes_geometry(device, fovy=30.0)
model = model.eval()

Some weights of ViTModel were not initialized from the model checkpoint at facebook/dino-vitb16 and are newly initialized: ['encoder.layer.0.adaLN_modulation.1.bias', 'encoder.layer.0.adaLN_modulation.1.weight', 'encoder.layer.1.adaLN_modulation.1.bias', 'encoder.layer.1.adaLN_modulation.1.weight', 'encoder.layer.10.adaLN_modulation.1.bias', 'encoder.layer.10.adaLN_modulation.1.weight', 'encoder.layer.11.adaLN_modulation.1.bias', 'encoder.layer.11.adaLN_modulation.1.weight', 'encoder.layer.2.adaLN_modulation.1.bias', 'encoder.layer.2.adaLN_modulation.1.weight', 'encoder.layer.3.adaLN_modulation.1.bias', 'encoder.layer.3.adaLN_modulation.1.weight', 'encoder.layer.4.adaLN_modulation.1.bias', 'encoder.layer.4.adaLN_modulation.1.weight', 'encoder.layer.5.adaLN_modulation.1.bias', 'encoder.layer.5.adaLN_modulation.1.weight', 'encoder.layer.6.adaLN_modulation.1.bias', 'encoder.layer.6.adaLN_modulation.1.weight', 'encoder.layer.7.adaLN_modulation.1.bias', 'encoder.layer.7.adaLN_modulation.1.w

In [10]:
input_cameras = get_zero123plus_input_cameras().to(device)

input_mesh = trimesh.load("tmp/mesh.obj")
rotation_matrix_a = trimesh.transformations.rotation_matrix(
    angle=np.radians(-90),
    direction=[1, 0, 0]
)
rotation_matrix_b = trimesh.transformations.rotation_matrix(
    angle=np.radians(-90),
    direction=[0, 1, 0]
)
input_mesh.apply_transform(rotation_matrix_a)
input_mesh.apply_transform(rotation_matrix_b)

input_cameras_numpy = input_cameras.cpu().numpy()
extrinsics = input_cameras_numpy[0, :, :12].reshape(-1, 3, 4)
fov = 30.0

images = []
for i, extrinsic in enumerate(extrinsics):
    extrinsic4x4 = np.eye(4)
    extrinsic4x4[0, :] = extrinsic[1, :]
    extrinsic4x4[1, :] = extrinsic[2, :]
    extrinsic4x4[2, :] = extrinsic[0, :]
    camera = trimesh.scene.Camera(fov=(fov, fov))
    scene = trimesh.Scene(
        input_mesh,
        camera=camera,
        camera_transform=extrinsic4x4
    )
    image = scene.save_image(resolution=[320, 320], visible=False)
    with open(f"tmp/mesh_{i}.png", "wb") as f:
        f.write(image)

    image = Image.open(io.BytesIO(image)).convert("RGB")
    image = np.array(image, dtype=np.float32) / 255.0
    image = image.transpose(2, 0, 1)
    images.append(image)

""" images = []
image_paths = [f"tmp/output_image_{i}.png" for i in range(6)]
for image_path in image_paths:
    image = Image.open(image_path).convert("RGB")
    image = np.array(image, dtype=np.float32) / 255.0
    image = image.transpose(2, 0, 1)
    images.append(image) """

images = np.stack(images, axis=0)
print(images.shape)
images_processed = torch.from_numpy(images)
images_processed = images_processed.unsqueeze(0).to(device)
images_processed = v2.functional.resize(
    images_processed, 
    320, 
    interpolation=3, 
    antialias=True
).clamp(0, 1)

(6, 3, 320, 320)


In [13]:
with torch.no_grad():
    planes = model.forward_planes(images_processed, input_cameras)
    mesh_path = "tmp/mesh_post.obj"
    mesh_out = model.extract_mesh(
        planes,
        use_texture_map=True,
        texture_resolution=1024,
    )
    vertices, faces, uvs, mesh_tex_idx, tex_map = mesh_out
    save_obj_with_mtl(
        vertices.data.cpu().numpy(),
        uvs.data.cpu().numpy(),
        faces.data.cpu().numpy(),
        mesh_tex_idx.data.cpu().numpy(),
        tex_map.permute(1, 2, 0).data.cpu().numpy(),
        mesh_path,
    )